# GEMM 跨平台迁移教学版

这个 notebook 用来配合第 8 章，演示同一份 GEMM 逻辑如何迁移到不同平台。

它的目标不是跑出真实 benchmark，而是把迁移顺序、记录字段和口径固定下来。

## 阅读顺序

1. 先看统一的 GEMM 逻辑
2. 再看不同平台需要改什么
3. 然后看 benchmark 记录模板
4. 最后看为什么记录字段要统一

## 最小迁移模型

`same algorithm -> same correctness check -> different target/layout/tile -> different benchmark record`

这句话是第 8 章的核心：算法逻辑尽量不动，平台参数显式分开。

In [ ]:
benchmark_record = {
    'scenario': 'cross-platform gemm migration',
    'operator': 'GEMM',
    'platform': 'NVIDIA / ROCm / Ascend',
    'target': 'cuda',
    'version': 'draft',
    'driver_version': 'unknown',
    'runtime_version': 'unknown',
    'dtype': 'float16',
    'shape': 'M=1024,N=1024,K=1024',
    'batch': 1,
    'warmup': 10,
    'runs': 100,
    'latency_ms': None,
    'throughput': None,
    'tflops': None,
    'gbps': None,
    'peak_mem_mb': None,
    'avg_mem_mb': None,
    'baseline': 'plain GEMM baseline',
    'optimization': 'tile + layout + backend tuning',
    'notes': 'fill this after running on each target',
}

for key in ['scenario', 'operator', 'platform', 'target', 'shape', 'dtype', 'warmup', 'runs']:
    print(f'{key}: {benchmark_record[key]}')

## 迁移时先看什么

- 先确认 target 能编译
- 再确认 shape 和 dtype 能跑通
- 然后只改最小必要的布局和 tile 参数
- 最后才做平台专有优化

这个顺序可以避免一开始就把多个变量混在一起。

## 记录表的作用

记录表不是附录，它是证据结构。

- 没有统一字段，就没法横向对比
- 没有统一 warmup / runs，就没法复测
- 没有统一 target / driver 信息，就没法解释差异